# PySpark - Fundamentos e aplicações

A ideia deste *notebook* é apresentar os fundamentos de uso da API *Python* para *Spark*. Vamos verificar os principais comandos, e entender o que são ações, transformações, execução *lazy* e as famosas DAG's geradas pelo *Spark* (essa última parte de uma forma mais superficial).

Tudo isso será feito dentro de um ambiente *Databricks*, o qual conta com algumas funcionalidades que facilitam nosso trabalho diário.

Após essa breve introdução e explanação dos conceitos, nós iremos simular uma utilização em um problema de negócio, para facilitar a assimilação de tudo que foi visto.

## Pontos importantes sobre essa interface

A aba na lateral direita nos permite:

- criar *notebooks*, a partir da opção *create*;
- importar dados, também na opção *create*. Essa importação é feita a partir de uma interface gráfica, muito simples de ser utilizada (vale a pena o teste);
- criação de um *cluster* na opção *compute*. Todo *notebook* deve ser associado a um *cluster* para execução.

## Dados utilizados

Os dados utilizados estão dispobilizados no *Kaggle*:

- Link para *download*: [Conjunto de dados](https://www.kaggle.com/olistbr/brazilian-ecommerce?select=olist_order_payments_dataset.csv)

Nesse projeto vamos utilizar as tabelas:

- *olist_customers_dataset.csv*;
- *olist_order_payments_dataset.csv*;
- *olist_orders_dataset.csv*.

### Contexto dos dados

"*This dataset was generously provided by Olist, the largest department store in Brazilian marketplaces. Olist connects small businesses from all over Brazil to channels without hassle and with a single contract. Those merchants are able to sell their products through the Olist Store and ship them directly to the customers using Olist logistics partners. After a customer purchases the product from Olist Store a seller gets notified to fulfill that order. Once the customer receives the product, or the estimated delivery date is due, the customer gets a satisfaction survey by email where he can give a note for the purchase experience and write down some comments.*"

#### Link entre os dados

![databases](https://i.imgur.com/HRhd2Y0.png)

## Referências úteis

- Documentação: [Documentação PySpark](https://spark.apache.org/docs/latest/api/python/index.html);
- Criando tabelas: [Criação de tabelas no databricks](https://docs.databricks.com/data/tables.html).

Mãos a obra
Agora que temos um conhecimento sobre os principais métodos dessa API, vamos aplicá-los para resolvermos alguns problemas de negócio.A área de negócio nos fez as seguintes indagações:

1 - Quantidade de ordens agrupadas por ANO/MÊS/STATUS (eles precisam de um arquivo .CSV contendo todas essas informações);

2 - Quantidade de usuários por estado (para identificarem onde precisam focar os esforços de marketing);

3 - Além da quantidade, a área de negócio precisa do ranking de cada estado (qual é o primeiro, segundo, etc) em termos da quantidade de usuários;

4 - Quantidade de usuários que tiveram mais de três ordens;

5 - Dos usuários que tiveram pelo menos três ordens, quantos dias isso (ter a terceira ordem) levou em relação a primeira ordem de compra;Devemos saber o seguinte sobre o conjunto de dados:
- customer_id: ID do cliente;
- customer_unique_id: ID único de cada cliente (esse ID engloba vários customer_id);
- order_purchase_timestamp: timestamp da data da ordem;
- customer_state: estado do cliente;payment_type: tipo de pagamento;

Endereço das BasesUtilizar bases:
- olist_customers_dataset.csv
- olist_orders_dataset.csv
- olist_order_payments_dataset.csv

In [1]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.4/281.4 MB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.0/199.0 KB 15.0 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.2.1-py2.py3-none-any.whl size=281853642 sha256=5a4328a5f5247b2d6a1b480fbff3aa44039206c4ca1c7ffd413ca12b03c8395b
  Stored in directory: /root/.cache/pip/wheels/9f/f5/07/7cd8017084dce4e93e84e92efd1e1d5334db05f2e83bcef74f
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.5
    Uninstalling py4j-0.10.9.5:
      Successfully uninstalled py4j-0.10.9.5


In [2]:
import pyspark.sql.types as T         #Define os tipos nativos do PySpark
import pyspark.sql.functions as F     #Importa as funções nativas do Spark para manipulação dos dados
from pyspark.sql.window import Window #Importa a função utilizada para criação de janelas

from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Analysis with pyspark").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
22/05/16 03:32:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
customers_df = spark.read.option("delimeter","|").csv('../input/brazilian-ecommerce/olist_customers_dataset.csv',header = True)
customers_df = customers_df.na.drop()
customers_df.printSchema()
customers_df.show()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                   09790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                   01151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                   08775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83.

In [4]:
orders_df = spark.read.option("delimeter","|").csv('../input/brazilian-ecommerce/olist_orders_dataset.csv',header = True)
orders_df = orders_df.withColumn('order_purchase_timestamp', F.to_timestamp('order_purchase_timestamp')).withColumn('order_delivered_carrier_data', F.to_timestamp('order_delivered_carrier_date')).withColumn('order_approved_at',F.to_timestamp('order_approved_at')).withColumn('order_delivered_customer_date',F.to_timestamp('order_delivered_customer_date')).withColumn('order_estimed_delivery_date',F.to_timestamp('order_estimated_delivery_date'))
orders_df.printSchema()
orders_df.show()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- order_delivered_carrier_data: timestamp (nullable = true)
 |-- order_estimed_delivery_date: timestamp (nullable = true)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+----------------------------+---------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_de

In [5]:
payment_df = spark.read.option("delimeter","|").csv('../input/brazilian-ecommerce/olist_order_payments_dataset.csv',header = True)
payment_df = payment_df.na.drop()
payment_df.printSchema()
payment_df.show()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| c

1 - Quantidade de ordens agrupadas por ANO/MÊS/STATUS (eles precisam de um arquivo .CSV contendo todas essas informações)

In [6]:
orders_df = orders_df.withColumn("order_year_month", F.date_format(F.col("order_purchase_timestamp"), format="y-M"))
answer_1 = orders_df.groupby("order_year_month", "order_status").count().withColumnRenamed('count','No_of_orders_year_month')
answer_1.show(3)

+----------------+------------+-----------------------+
|order_year_month|order_status|No_of_orders_year_month|
+----------------+------------+-----------------------+
|          2017-3|   delivered|                   2546|
|          2018-6|   delivered|                   6099|
|          2017-6|   delivered|                   3135|
+----------------+------------+-----------------------+
only showing top 3 rows



2 - Quantidade de usuários por estado (para identificarem onde precisam focar os esforços de marketing);

In [7]:
answer_2 = customers_df.groupby("customer_state").count().withColumnRenamed('count','No_of_customers_state')
answer_2.show(2)

+--------------+---------------------+
|customer_state|No_of_customers_state|
+--------------+---------------------+
|            SC|                 3637|
|            RO|                  253|
+--------------+---------------------+
only showing top 2 rows



3 - Além da quantidade, a área de negócio precisa do ranking de cada estado (qual é o primeiro, segundo, etc) em termos da quantidade de usuários;

In [8]:
answer_3 = answer_2.orderBy('No_of_customers_state',ascending=False)
answer_3 = answer_3.withColumn('rank',F.monotonically_increasing_id()+1)
answer_3.show()

+--------------+---------------------+----+
|customer_state|No_of_customers_state|rank|
+--------------+---------------------+----+
|            SP|                41746|   1|
|            RJ|                12852|   2|
|            MG|                11635|   3|
|            RS|                 5466|   4|
|            PR|                 5045|   5|
|            SC|                 3637|   6|
|            BA|                 3380|   7|
|            DF|                 2140|   8|
|            ES|                 2033|   9|
|            GO|                 2020|  10|
|            PE|                 1652|  11|
|            CE|                 1336|  12|
|            PA|                  975|  13|
|            MT|                  907|  14|
|            MA|                  747|  15|
|            MS|                  715|  16|
|            PB|                  536|  17|
|            PI|                  495|  18|
|            RN|                  485|  19|
|            AL|                

4 - Quantidade de usuários que tiveram mais de três ordens;

In [9]:
orders_customers = orders_df.join(customers_df, on="customer_id", how="left")
answer_4 = (orders_customers.groupby("customer_unique_id").count().where(F.col("count") >= 3))

print(f"Quantidade de clientes: {answer_4.count()}.")

Quantidade de clientes: 252.


https://www.geeksforgeeks.org/groupby-and-filter-data-in-pyspark/

5 - Dos usuários que tiveram pelo menos três ordens, quantos dias isso (ter a terceira ordem) levou em relação a primeira ordem de compra;